# Creator crossover across tracked titles

**This measures creator crossover, not viewer overlap — read every result
below with that distinction in mind, not as an answer to the brief's
Limitation 1 (multi-homing).** Limitation 1 asks whether the *same
viewers* watch multiple titles. This notebook instead asks whether the
*same streamers* broadcast multiple titles at an above-threshold audience
size. Those are related but not the same question: a streamer switching
between two games says their own following is willing to watch them play
either — informative, but it's a proxy for creator supply, not viewer
demand. It's one inferential step removed from what Limitation 1 actually
asks, not a direct measurement of it.

**Method:** `viewership_snapshots` classifies every captured stream as
either `is_official_broadcast=1` (a curated official channel,
`config/channels.yaml`) or `0` (captured because it crossed the
above-threshold viewer-count cutoff — "tier 2" in `collectors/
twitch_poll.py`'s own capture-tier language; tier 3, below threshold, is
never captured per-stream at all). This notebook looks only at tier-2
(`is_official_broadcast=0`) rows: does a given `channel_id` appear as an
above-threshold streamer for more than one tracked title, and if so, how
is its above-threshold viewer-time split between them. `config/
channels.yaml` is empty as of this run (nothing curated yet), so every
captured row is currently tier-2 by construction — the official-channel
filter is a no-op today but the correct filter to keep, since it'll start
doing real work once that config is populated.

**This only covers time since the Twitch collector started
(2026-08-31)** — a few days as of this run. Early results are thin by
construction, not a defect in the method: a channel needs to be caught
mid-stream, above threshold, for more than one title within this short a
window, which is a real but incomplete sample of actual crossover
activity. No need to wait for a bigger window before running this — read
today's output as directional and expect it to firm up over the
following weeks as more data accumulates.

In [1]:
# Imports and repo path setup — same pattern as the other notebooks.
import sys
from itertools import combinations
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import yaml

from etl.db import get_connection

In [2]:
# All 23 active titles — config/titles.yaml is the source of truth, same
# as every other notebook in this project.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
len(active_titles)

23

## Load tier-2 (above-threshold, non-official) stream records

One row per full-detail stream per poll. `viewer_count` summed per
`(channel_id, title_id)` gives a viewer-time figure, same convention used
in `notebooks/niche_membership.ipynb`'s language-mix analysis — weighted
by how long and how big the stream was, not just by poll count.

In [3]:
conn = get_connection()
window_start, window_end = conn.execute(
    "SELECT MIN(captured_at), MAX(captured_at) FROM viewership_snapshots"
).fetchone()
print(f"viewership_snapshots window: {window_start} to {window_end}")

raw_rows = conn.execute(
    """
    SELECT channel_id, channel_login, title_id, viewer_count
    FROM viewership_snapshots
    WHERE is_official_broadcast = 0
    """
).fetchall()
conn.close()

raw_df = pd.DataFrame(raw_rows, columns=["channel_id", "channel_login", "title_id", "viewer_count"])

# Login for display only, kept separate from the aggregation below — a
# channel_login can change mid-window if a streamer renames, and grouping
# by it directly would silently split one channel's viewer-time across
# what looks like two different rows. channel_id is the only real join key.
login_by_channel = raw_df.groupby("channel_id")["channel_login"].agg(lambda s: s.mode().iat[0])

channel_title_df = (
    raw_df.groupby(["channel_id", "title_id"])["viewer_count"].sum().rename("viewer_time").reset_index()
)

print(f"{channel_title_df['channel_id'].nunique()} distinct above-threshold, non-official channels seen")
print(f"{len(channel_title_df)} (channel, title) combinations total")

viewership_snapshots window: 2026-08-31T10:13:48Z to 2026-09-04T09:25:58Z


85735 distinct above-threshold, non-official channels seen
89656 (channel, title) combinations total


## Crossover channels

A channel counts as crossover if it appears above-threshold for more
than one tracked title anywhere in the window — no minimum viewer-time
cutoff at this stage, so this first count includes channels that barely
crossed the threshold once for a second title. The per-channel table
below carries actual viewer-time so a marginal crossover is visibly
marginal, not hidden.

In [4]:
titles_per_channel = channel_title_df.groupby("channel_id")["title_id"].nunique()
crossover_channel_ids = titles_per_channel[titles_per_channel > 1].index

crossover_df = channel_title_df[channel_title_df["channel_id"].isin(crossover_channel_ids)].copy()
crossover_df["channel_login"] = crossover_df["channel_id"].map(login_by_channel)
crossover_df["display_name"] = crossover_df["title_id"].map(display_name_by_id)

channel_totals = crossover_df.groupby("channel_id")["viewer_time"].sum()
crossover_df["share_of_channel_viewer_time_pct"] = (
    crossover_df["viewer_time"] / crossover_df["channel_id"].map(channel_totals) * 100
).round(1)

print(f"{len(crossover_channel_ids)} channels (of {channel_title_df['channel_id'].nunique()} total) "
      f"streamed above-threshold for more than one title this window.")
print("\nDistribution of how many titles a crossover channel touched:")
print(titles_per_channel[titles_per_channel > 1].value_counts().sort_index())

3715 channels (of 85735 total) streamed above-threshold for more than one title this window.

Distribution of how many titles a crossover channel touched:
title_id
2    3517
3     190
4       8
Name: count, dtype: int64


## Per-channel breakdown, top 30 by total above-threshold viewer-time

Long format (one row per channel × title), not a wide pivot — most
crossover channels only touch 2 of 23 titles, so a full title-by-title
grid would be almost entirely empty. Sorted by each channel's total
viewer-time across all its titles, so the crossover activity that
actually moved meaningful audience shows up first, ahead of channels that
crossed the threshold once for a handful of viewers.

In [5]:
TOP_N_CHANNELS = 30

top_channel_ids = channel_totals.sort_values(ascending=False).head(TOP_N_CHANNELS).index

top_display = (
    crossover_df[crossover_df["channel_id"].isin(top_channel_ids)]
    .assign(channel_total_viewer_time=lambda d: d["channel_id"].map(channel_totals))
    .sort_values(["channel_total_viewer_time", "channel_id", "viewer_time"], ascending=[False, True, False])
    [["channel_login", "display_name", "viewer_time", "share_of_channel_viewer_time_pct", "channel_total_viewer_time"]]
    .rename(columns={"display_name": "title"})
    .reset_index(drop=True)
)
top_display

,channel_login,title,viewer_time,share_of_channel_viewer_time_pct,channel_total_viewer_time
0,stylishnoob4,Apex Legends,33535,67.7,49507
1,stylishnoob4,Street Fighter 6,15972,32.3,49507
2,itachi,VALORANT,37619,83.6,45003
3,itachi,Rocket League,7384,16.4,45003
4,aussieantics,Fortnite,39954,94.5,42269
...,...,...,...,...,...
56,c_a_k_e,PUBG: BATTLEGROUNDS,2017,35.3,5719
57,vadeal,VALORANT,4571,82.1,5566
58,vadeal,Fortnite,995,17.9,5566
59,goodoq,Dota 2,4827,89.7,5380


## Bonus rollup: which title pairs share the most crossover channels

Not separately requested, but a direct one-line-of-code aggregation of
the same table above — for each pair of titles, how many distinct
channels streamed both above-threshold this window. Read this the same
way as everything else here: a count of *creators* two titles share, not
a measurement of shared *audience*.

In [6]:
titles_by_channel = crossover_df.groupby("channel_id")["title_id"].apply(set)

pair_counts = {}
for title_set in titles_by_channel:
    for a, b in combinations(sorted(title_set), 2):
        pair_counts[(a, b)] = pair_counts.get((a, b), 0) + 1

pair_rollup = pd.DataFrame(
    [{"title_a": display_name_by_id[a], "title_b": display_name_by_id[b], "shared_crossover_channels": n}
     for (a, b), n in pair_counts.items()]
).sort_values("shared_crossover_channels", ascending=False).reset_index(drop=True)

print(f"{len(pair_rollup)} title pairs share at least one crossover channel")
pair_rollup.head(20)

141 title pairs share at least one crossover channel


,title_a,title_b,shared_crossover_channels
0,League of Legends,VALORANT,462
1,League of Legends,Teamfight Tactics,298
2,Overwatch,VALORANT,263
3,Apex Legends,VALORANT,229
4,Fortnite,VALORANT,203
5,Fortnite,Overwatch,167
6,Counter-Strike 2,VALORANT,163
7,Teamfight Tactics,VALORANT,147
8,Apex Legends,Overwatch,138
9,Fortnite,Rocket League,132


## Caveats

- **Creator crossover, not viewer overlap — restated, not a footnote.**
  Nothing above measures whether the same viewers watch two titles. A
  streamer appearing above-threshold for two titles says that streamer's
  own audience followed them across both at that moment; it doesn't say
  anything about the other 999 viewers of either title who never saw the
  other one. Treat this as a lower bound on *some* cross-title audience
  movement, driven by creators, not as a measure of overlap between the
  titles' audiences as a whole.
- **This is a few days of data, not a settled base rate.** The
  above-threshold cutoff means a channel needs a real audience spike to
  register for a second title at all — plenty of genuine crossover
  streamers (who play both games regularly but at modest, below-threshold
  audience for one of them) won't show up yet, or ever, under this
  method. Re-run as `viewership_snapshots` accumulates; don't treat
  today's channel count or pair-rollup numbers as final.
- **`is_official_broadcast` is a no-op filter today**, not a validated
  one — `config/channels.yaml` is empty, so nothing has actually been
  excluded by it yet. Once official channels are curated, re-check that
  this notebook's crossover counts don't include large media-org channels
  broadcasting several titles' tournaments (that's a different phenomenon
  than an individual creator crossing over) rather than only genuine
  creator-driven crossover.
- **A channel_id is a Twitch account, not necessarily one person** — some
  crossover channels here may be multi-game variety orgs, co-streaming
  services, or rebroadcast channels rather than a single creator. Worth a
  manual look at the top of the per-channel table before treating any
  individual name as a "this creator bridges these two fandoms" finding.